# Quickstart: 4-bit Base Case Validation
**Runtime → Change runtime type → T4 GPU**

In [ ]:
# Cell 1: Setup (safe to re-run)
import os
if not os.path.exists('/content/recursive-transformers/src'):
    !git clone https://github.com/dhruvsyam123/recursive-transformers.git /content/recursive-transformers
else:
    !cd /content/recursive-transformers && git pull
%cd /content/recursive-transformers
!pip install -q jax[cuda12] equinox optax jaxtyping numpy pyyaml matplotlib einops wandb

import jax
print(f"JAX devices: {jax.devices()}")
print(f"JAX backend: {jax.default_backend()}")

In [ ]:
# Cell 2: Build 4-bit dataset — ALL 256 pairs (exhaustive, no split)
from src.data.karatsuba_trace import KaratsubaTraceGenerator
from src.data.dataset import MultiplicationDataset, DataConfig

gen = KaratsubaTraceGenerator(base_case_bits=4)
trace = gen.generate(179, 214, 8)
print(f"179 * 214 = {trace.trace_product} (correct: {trace.verify()})\n")

# Use ALL 256 pairs for training — base case must be memorized perfectly
config = DataConfig(
    bit_widths=[4],
    algorithm='karatsuba',
    base_case_bits=4,
    train_fraction=1.0,  # all pairs for training
)
ds = MultiplicationDataset(config)
print(ds.summary())

In [ ]:
# Cell 3: Train 4-bit base case
from src.model.looped_transformer import create_model, count_parameters
from src.data.tokenizer import Tokenizer
import jax
import jax.numpy as jnp
import equinox as eqx
import optax

tok = Tokenizer()
# Token IDs for tags — tokens after [INPUT] up to the next tag are unpredictable input data
INPUT_ID = tok.encode_token("[INPUT]")
BASE_MUL_ID = tok.encode_token("[BASE_MUL]")

key = jax.random.PRNGKey(42)
model = create_model(
    key=key,
    vocab_size=143,
    d_model=128,
    n_heads=4,
    d_ff=256,
    n_shared_layers=1,
    max_loop_iterations=8,
    max_seq_len=32,
    position_encoding_type='sinusoidal',
)

N_LOOPS = 4
print(f"Model parameters: {count_parameters(model):,}")

schedule = optax.warmup_cosine_decay_schedule(
    init_value=1e-5, peak_value=3e-3, warmup_steps=100,
    decay_steps=5000, end_value=1e-5
)
optimizer = optax.adamw(learning_rate=schedule, weight_decay=0.01)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

def make_predictable_mask(input_ids, pad_mask):
    """Mask out input data bits (between [INPUT] and [BASE_MUL]) from loss.
    Only compute loss on tokens the model can actually predict."""
    batch_size, seq_len = input_ids.shape
    # For each sequence, find [INPUT] and [BASE_MUL] positions
    # Tokens between them are unpredictable input data
    is_input_tag = (input_ids == INPUT_ID)
    is_base_tag = (input_ids == BASE_MUL_ID)
    # Cumsum trick: after [INPUT], we're in input region; after [BASE_MUL], we're out
    in_input_region = (jnp.cumsum(is_input_tag, axis=1) > 0) & (jnp.cumsum(is_base_tag, axis=1) == 0)
    # Don't mask the [INPUT] tag itself (it's predictable), just the data bits after it
    predictable = ~in_input_region | is_input_tag
    return pad_mask * predictable.astype(jnp.float32)

@eqx.filter_jit
def train_step(model, opt_state, input_ids, target_ids, mask):
    def loss_fn(model):
        def forward(ids):
            positions = jnp.arange(ids.shape[0])
            return model(ids, positions, n_loops=N_LOOPS)
        logits = jax.vmap(forward)(input_ids)
        log_probs = jax.nn.log_softmax(logits, axis=-1)
        targets_one_hot = jax.nn.one_hot(target_ids, logits.shape[-1])
        token_losses = -jnp.sum(targets_one_hot * log_probs, axis=-1)
        return jnp.sum(token_losses * mask) / jnp.maximum(jnp.sum(mask), 1.0)
    loss, grads = eqx.filter_value_and_grad(loss_fn)(model)
    updates, opt_state_new = optimizer.update(
        grads, opt_state, eqx.filter(model, eqx.is_array)
    )
    model_new = eqx.apply_updates(model, updates)
    return model_new, opt_state_new, loss

print("Training 4-bit base case (4 loops, d=128)...")
for epoch in range(500):
    epoch_loss = 0
    n_batches = 0
    for batch in ds.get_batch(split='train', batch_size=32):
        input_ids = jnp.array(batch['input_ids'])
        target_ids = jnp.array(batch['target_ids'])
        pad_mask = jnp.array(batch['input_mask'])
        # Mask out unpredictable input bits from loss
        loss_mask = make_predictable_mask(input_ids, pad_mask)
        model, opt_state, loss = train_step(model, opt_state, input_ids, target_ids, loss_mask)
        epoch_loss += float(loss)
        n_batches += 1
    if epoch % 50 == 0:
        print(f"Epoch {epoch:4d} | Loss: {epoch_loss/n_batches:.4f}")

print("Done!")

In [ ]:
# Cell 4: Evaluate 4-bit accuracy on ALL 256 pairs
correct = 0
total = 0
for batch in ds.get_batch(split='train', batch_size=64, shuffle=False):
    input_ids = jnp.array(batch['input_ids'])
    target_ids = jnp.array(batch['target_ids'])
    pad_mask = jnp.array(batch['input_mask'])
    eval_mask = make_predictable_mask(input_ids, pad_mask)

    def forward(ids):
        positions = jnp.arange(ids.shape[0])
        return model(ids, positions, n_loops=N_LOOPS)
    logits = jax.vmap(forward)(input_ids)
    preds = jnp.argmax(logits, axis=-1)

    matches = (preds == target_ids) | (eval_mask == 0)
    correct += int(jnp.all(matches, axis=-1).sum())
    total += len(batch['x'])

print(f"4-bit multiplication accuracy: {correct}/{total} = {correct/total:.1%}")
print()
if correct / total >= 0.99:
    print("PASS — base case learned. Ready for 8-bit Karatsuba.")
else:
    print(f"Needs more training ({total - correct} pairs wrong). Paste output back to Claude.")

---
## Phase 2: 8-bit Karatsuba Training
If 4-bit passed, run the cells below to train on 8-bit Karatsuba traces (1 level of recursion) and test length generalization to 16-bit.

In [ ]:
# Cell 6: Build 8-bit Karatsuba dataset
from src.data.dataset import MultiplicationDataset, DataConfig

config_8bit = DataConfig(
    bit_widths=[4, 8],  # curriculum: 4-bit base + 8-bit Karatsuba
    algorithm='karatsuba',
    base_case_bits=4,
    ordering='depth_first',
    max_samples_per_width=4000,
    seed=42,
)
ds8 = MultiplicationDataset(config_8bit)
print(ds8.summary())
print(f"\nSample 8-bit trace length: {ds8.train_examples[0]['seq_len']} tokens")

In [ ]:
# Cell 7: Train 8-bit Karatsuba model
key = jax.random.PRNGKey(123)
model8 = create_model(
    key=key,
    vocab_size=143,
    d_model=256,
    n_heads=8,
    d_ff=512,
    n_shared_layers=1,
    max_loop_iterations=16,
    max_seq_len=512,
    position_encoding_type='sinusoidal',
)

N_LOOPS_8 = 8
print(f"Model parameters: {count_parameters(model8):,}")

schedule8 = optax.warmup_cosine_decay_schedule(
    init_value=1e-5, peak_value=1e-3, warmup_steps=500,
    decay_steps=30000, end_value=1e-5
)
optimizer8 = optax.adamw(learning_rate=schedule8, weight_decay=0.01)
opt_state8 = optimizer8.init(eqx.filter(model8, eqx.is_array))

# Need to handle variable-length tags for 8-bit traces (has [SPLIT], [SUB_MUL_*], etc.)
SPLIT_ID = tok.encode_token("[SPLIT]")

def make_predictable_mask_8bit(input_ids, pad_mask):
    """For 8-bit Karatsuba: mask out all raw input data bits at every recursion level.
    Input bits follow [INPUT] or [SUB_MUL_*] tags; predictable tokens follow
    computation tags like [BASE_MUL], [SPLIT], [ADD], [SUB], [COMBINE], [OUTPUT]."""
    # Simple approach: mask positions where target is a bit (0 or 1)
    # that directly follows an [INPUT] tag before any computation tag.
    # For now, use the same approach: mask between [INPUT] and next non-input tag.
    # This works because in depth-first traces, each [INPUT] is followed by its
    # operand bits, then either [BASE_MUL] or [SPLIT].
    BIT_0 = tok.encode_token(0)
    BIT_1 = tok.encode_token(1)
    is_bit = (input_ids == BIT_0) | (input_ids == BIT_1)
    # A bit is "input data" if no computation tag has appeared since the last [INPUT]
    # We'll use a simpler heuristic: only mask bits in the very first [INPUT] section
    # For recursive traces, intermediate inputs are derived (predictable in principle)
    # So just mask the top-level input bits
    input_tag_id = tok.encode_token("[INPUT]")
    # Find first [INPUT] tag position per sequence
    is_first_input = (jnp.cumsum(input_ids == input_tag_id, axis=1) == 1) & (input_ids != input_tag_id)
    # Check if we're still in the first input section (before first non-bit token after [INPUT])
    # Approximate: mask first 2*n_bits tokens after [INPUT] (for 8-bit: 16 data bits)
    # More robust: mask bits that appear before the first [SPLIT] or [BASE_MUL]
    has_seen_split = jnp.cumsum((input_ids == SPLIT_ID) | (input_ids == BASE_MUL_ID), axis=1) > 0
    in_top_input = ~has_seen_split & is_bit
    predictable = ~in_top_input
    return pad_mask * predictable.astype(jnp.float32)

@eqx.filter_jit
def train_step_8bit(model, opt_state, input_ids, target_ids, mask):
    def loss_fn(model):
        def forward(ids):
            positions = jnp.arange(ids.shape[0])
            return model(ids, positions, n_loops=N_LOOPS_8)
        logits = jax.vmap(forward)(input_ids)
        log_probs = jax.nn.log_softmax(logits, axis=-1)
        targets_one_hot = jax.nn.one_hot(target_ids, logits.shape[-1])
        token_losses = -jnp.sum(targets_one_hot * log_probs, axis=-1)
        return jnp.sum(token_losses * mask) / jnp.maximum(jnp.sum(mask), 1.0)
    loss, grads = eqx.filter_value_and_grad(loss_fn)(model)
    updates, opt_state_new = optimizer8.update(
        grads, opt_state, eqx.filter(model, eqx.is_array)
    )
    model_new = eqx.apply_updates(model, updates)
    return model_new, opt_state_new, loss

# Mount Google Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
CKPT_DIR = '/content/drive/MyDrive/karatsuba_checkpoints/'
os.makedirs(CKPT_DIR, exist_ok=True)

print("Training 8-bit Karatsuba (8 loops, d=256)...")
for epoch in range(200):
    epoch_loss = 0
    n_batches = 0
    for batch in ds8.get_batch(split='train', batch_size=32, max_len=400):
        input_ids = jnp.array(batch['input_ids'])
        target_ids = jnp.array(batch['target_ids'])
        pad_mask = jnp.array(batch['input_mask'])
        loss_mask = make_predictable_mask_8bit(input_ids, pad_mask)
        model8, opt_state8, loss = train_step_8bit(model8, opt_state8, input_ids, target_ids, loss_mask)
        epoch_loss += float(loss)
        n_batches += 1
    if epoch % 20 == 0:
        avg_loss = epoch_loss / n_batches
        print(f"Epoch {epoch:4d} | Loss: {avg_loss:.4f} | Batches: {n_batches}")
    # Checkpoint every 50 epochs
    if epoch % 50 == 0 and epoch > 0:
        ckpt_path = os.path.join(CKPT_DIR, f'model8_epoch{epoch}.eqx')
        eqx.tree_serialise_leaves(ckpt_path, model8)
        print(f"  Saved checkpoint: {ckpt_path}")

# Save final
eqx.tree_serialise_leaves(os.path.join(CKPT_DIR, 'model8_final.eqx'), model8)
print("Done! Final checkpoint saved.")

In [ ]:
# Cell 8: Evaluate 8-bit on training length + test 16-bit generalization
from src.data.dataset import DataConfig, MultiplicationDataset

def evaluate_at_width(model, n_bits, n_loops, n_samples=200):
    """Evaluate model on a specific bit width."""
    eval_config = DataConfig(
        bit_widths=[n_bits], algorithm='karatsuba', base_case_bits=4,
        max_samples_per_width=n_samples, train_fraction=0.0, seed=999,
    )
    eval_ds = MultiplicationDataset(eval_config)
    max_len = eval_ds.get_max_seq_len('test') + 1

    correct = 0
    total = 0
    for batch in eval_ds.get_batch(split='test', batch_size=32, shuffle=False, max_len=max_len):
        input_ids = jnp.array(batch['input_ids'])
        target_ids = jnp.array(batch['target_ids'])
        pad_mask = jnp.array(batch['input_mask'])
        eval_mask = make_predictable_mask_8bit(input_ids, pad_mask)

        def forward(ids):
            positions = jnp.arange(ids.shape[0])
            return model(ids, positions, n_loops=n_loops)
        logits = jax.vmap(forward)(input_ids)
        preds = jnp.argmax(logits, axis=-1)
        matches = (preds == target_ids) | (eval_mask == 0)
        correct += int(jnp.all(matches, axis=-1).sum())
        total += len(batch['x'])

    return correct, total

print("Length generalization evaluation:")
print(f"{'Bits':>6} {'Loops':>6} {'Accuracy':>12}")
print("-" * 28)
for n_bits, n_loops in [(8, 8), (16, 12), (32, 16)]:
    try:
        c, t = evaluate_at_width(model8, n_bits, n_loops, n_samples=200)
        print(f"{n_bits:6d} {n_loops:6d} {c:4d}/{t:4d} = {c/t:.1%}")
    except Exception as e:
        print(f"{n_bits:6d} {n_loops:6d} ERROR: {e}")